In [23]:
"""
Main training script for BINN, using DistributedDataParallel.

See run_binn_interactive.sh for example usage with reasonable hyperparameters.
"""
import csv
import functools
import math
import sys
import time
import random
import warnings
import subprocess
import argparse
from collections import OrderedDict
import misc_utils
from sklearn.model_selection import KFold
from mlp import ConstantParameters
from pe_gcn_model import GridCellSpatialRelationEncoder
from torch.optim.swa_utils import AveragedModel, SWALR
from spatial_utils import *
from losses import binns_loss, compute_param_matching_loss, compute_param_violation_loss, compute_unconstrained_param_loss
import visualization_utils

# sys.path.append('C:/Users/hx293/Research_Data/BINN/')
# sys.path.append('/glade/u/home/haodixu/BINN')
# sys.path.append(r'/User/homes/ftao/Projects/BINNS/src_binns')

# Set HDF5_DISABLE_VERSION_CHECK to suppress version mismatch error
import os
os.environ['HDF5_DISABLE_VERSION_CHECK'] = '2'

from datetime import datetime, timedelta
import pandas as pd
from pandas import DataFrame as df
import numpy as np
from scipy.interpolate import pchip_interpolate

print("Start binns_DDP")

# @joshuafan: previously we set default dtype to float64 to avoid underflow in process-based model.
# Now checking float32 with fixed process-based model.
torch.set_default_dtype(torch.float32)

# Temporary hack to avoid printing np.float64(...) when printing out numpy scalars.
# TODO fix this
np.set_printoptions(legacy="1.21")

import os
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import multiprocessing
# from torch.multiprocessing import Process  # TODO CHECK
from multiprocessing import Process
from scipy.io import loadmat
import netCDF4 as ncread 
import mat73
from matplotlib import pyplot as plt

# LibMTL is a library for advanced multi-task loss weighting methods.
# Commenting these out as they are not essential for BINN training.
# import LibMTL.weighting as weighting_method
# import LibMTL.architecture as architecture_method

###################################
# Import CLM5 process-based model #
###################################
# fun_model_simu predicts at user-specified depths. fun_model_prediction predicts at 20 default layers.
from fun_matrix_COMPAS_Hardy import fun_model_simu, fun_model_prediction

# KAN Model
import pykan_josh

device = 'cpu'

Start binns_DDP


In [21]:
## Input data
job_id = '20250418-171608_BINN_Global_COMPAS_with_KAN'

## Dummy args input
dummy_args = argparse.Namespace()
dummy_args.seed = 111
dummy_args.model = 'kan'
dummy_args.dropout_prob = 0
dummy_args.param_constraint = 'hardsigmoid'
dummy_args.losses = ["smooth_l1", "param_reg", "param_violation", "kan_l1", "kan_entropy", "kan_coefdiff"]
# KAN
dummy_args.kan_grid = 10
dummy_args.kan_grid_margin = 1.0
dummy_args.kan_noise = 0.3
dummy_args.kan_base_fun = "identity"
dummy_args.kan_affine_trainable = True
dummy_args.categorical = 'embedding'

# default
dummy_args.vertical_mixing = 'original'
dummy_args.vectorized = 'yes'
dummy_args.pos_enc = 'none'
dummy_args.use_bn = False
dummy_args.activation = 'leaky_relu'
dummy_args.min_temp = 10
dummy_args.max_temp = 109
dummy_args.init = "xavier_uniform"
dummy_args.width = 128
dummy_args.num_layers = 1
dummy_args.residual = False
dummy_args.feature_dropout = 0
dummy_args.standardize_input = False
dummy_args.standardize_output = False



para_names = ['diffus', 'cryo', 'q10', 'efolding', 'taucwd', 'taul1', 'taul2', 'tau4doc', 'tau4mic', 'tau4poc', 'tau4maom','fl1_MIC', 'fl2_MIC', 'fMIC_DOC', 'fMIC_POC', 'fDOC_MAOM', 'CUEl1', 'CUEl2', 'CUEDOC', 'w-scaling', 'beta']
para_index = np.arange(len(para_names))
var4nn = ["BIO1", "BIO12", \
	"Clay_Content_avg", "Sand_Content_avg", \
	"Bulk_Density_avg", "SWC_v_Wilting_Point_avg", "pH_Water_avg", "CEC_avg", "cesm2_npp", "cesm2_vegc", \
	'Ald_avg', 'Alo_avg',\
	'Fed_avg', 'Feo_avg', \
	]



def set_seeds(seed):
	"""
	Attempts to set all random seeds to improve reproducibility.
	"""
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = True

# Set seeds for reproducibility
set_seeds(dummy_args.seed)

In [22]:
# Recreate the model
# Create a dummy model to get the input size
train_x = np.zeros((1, 20, 12, len(var4nn)))
train_y = np.zeros((1, 3, 200))
var_idx_to_emb = OrderedDict()
model_class, model_kwargs = misc_utils.get_model(dummy_args, var4nn, var_idx_to_emb, device,
														para_index, train_x, train_y)
model = model_class(**model_kwargs).to(device)
print(model)
new_checkpoint = torch.load('D:/BINN/OUTPUT_DATA/neural_network/20250418-142035_BINN_Global_COMPAS_with_KAN/opt_nn_20250418-142035_BINN_Global_COMPAS_with_KAN.pt', map_location=device, weights_only=False)
model.load_state_dict(new_checkpoint['model_state_dict'])

checkpoint directory created: ./model
saving model version 0.0
mlp_wrapper(
  (var_idx_to_emb): ModuleDict()
  (mlp): MultKAN(
    (act_fun): ModuleList(
      (0): KANLayer(
        (base_fun): Identity()
      )
    )
    (base_fun): Identity()
    (symbolic_fun): ModuleList(
      (0): Symbolic_KANLayer()
    )
  )
  (sigmoid): Hardsigmoid()
)


<All keys matched successfully>

In [25]:
model.plot()

AttributeError: 'mlp_wrapper' object has no attribute 'plot'

In [ ]:
# Plot the KAN model
PLOT_DIR = os.path.join('D:/BINN/OUTPUT_DATA/neural_network', job_id, 'visualizations')
# Produce edge/node importance scores
model.mlp.attribute()
model.mlp.node_attribute()

# Plot the unpruned model
model.mlp.plot(folder=os.path.join(PLOT_DIR, "splines"), in_vars=var4nn, out_vars=para_names, scale=5, varscale=0.1)
plt.savefig(os.path.join(PLOT_DIR, f"Best_model_kan_plot.png"))
plt.close()

# Plot the pruned model
pruned_model = model.mlp.prune(node_th=0.03, edge_th=0.03)
pruned_model.plot(folder=os.path.join(PLOT_DIR, "splines"), in_vars=var4nn, out_vars=para_names, scale=5, varscale=0.1)
plt.savefig(os.path.join(PLOT_DIR, f"Best_model_kan_plot_pruned.png"))
plt.close()


Exception: model hasn't seen any data yet.